# Fine-Tuning Tied-Weights Autoencoder on External scRNA-seq Data

This notebook loads a pre-trained sparse autoencoder model trained on the CELLxGENE 60M dataset,
aligns external gene expression matrices to the model's gene space, fine-tunes the model,
and saves before/after embeddings.

- Autoencoder: Single-layer, tied weights (Wᵗ encoder / W decoder)
- Latent size: 256
- Output: Latent embeddings before and after fine-tuning (`.npy`), and model checkpoint


In [28]:
# IMPORT LIBRARIES

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import scipy.sparse as sp

In [29]:
# LOAD MODEL GENE LIST

# Path to the CellCensus model gene metadata
model_gene_path = "/mnt/projects/debruinz_project/bisholea/capstone/60M_human_gene_metadata.pkl"

# Load model gene metadata
gene_metadata = pd.read_pickle(model_gene_path)

# Standardize gene names
model_genes = [str(g).strip().split(".")[0].upper() for g in gene_metadata["feature_name"] if pd.notna(g)]

print("Loaded model gene list")
print("Total number of model genes:", len(model_genes))

Loaded model gene list
Total number of model genes: 61888


In [30]:
# DEFINE TIED-WEIGHTS AUTOENCODER 

class Autoencoder(nn.Module):
    """
    Single-layer autoencoder with tied weights for gene expression data.

    Architecture:
        Input (x) → Linear encoder (Wᵗ) → Hidden (256) → Linear decoder (W) → Output (x̂)

    Parameters:
        input_dim (int): Number of input genes.
        latent_dim (int): Size of the latent representation.
    """
    def __init__(self, input_dim, latent_dim=256):
        super(Autoencoder, self).__init__()

        # Encoder: linear projection to latent space (no bias, no activation)
        self.encoder = nn.Linear(input_dim, latent_dim, bias=False)

    def forward(self, x):
        # Project input to latent space
        h = self.encoder(x) # H = X W^T

        # Reconstruct input from latent space using tied weights (h @ Wᵗ)
        x_hat = torch.matmul(h, self.encoder.weight) # X̂ = H W
        return x_hat, h
    
print("Defined Autoencoder class with tied weights")

Defined Autoencoder class with tied weights


In [31]:
# DEFINE HELPER FUNCTIONS

def clean_and_filter_genes(gene_list, X_matrix):
    """
    Clean gene names and drop columns with invalid (NaN or empty) names.
    
    Parameters:
        gene_list: list-like of raw gene names
        X_matrix: 2D numpy array (cells x genes) or scipy sparse matrix

    Returns:
        cleaned_genes: list of cleaned gene names
        filtered_X: matrix with columns corresponding to valid genes
    """
    cleaned_genes = []
    valid_indices = []

    for i, g in enumerate(gene_list):
        if pd.notna(g):  # not NaN
            g_clean = str(g).strip().split(".")[0].upper()
            if g_clean:  # not empty after cleaning
                cleaned_genes.append(g_clean)
                valid_indices.append(i)

    # Subset matrix columns to only valid gene names
    if sp.issparse(X_matrix):
        filtered_X = X_matrix[:, valid_indices]
    else:
        filtered_X = X_matrix[:, valid_indices]

    return cleaned_genes, filtered_X


def align_input_matrix(X_external, external_genes, model_genes):
    """
    Align external expression matrix to match the gene ordering used by the trained model.

    Parameters:
        X_external (np.ndarray): External expression data of shape (cells, genes).
        external_genes (list[str]): Gene names corresponding to columns in X_external.
        model_genes (list[str]): Gene names expected by the model (used for training).

    Returns:
        np.ndarray: Aligned expression matrix of shape (cells, len(model_genes)),
                    where missing genes are filled with zeros.
    """
     # Map each external gene to its index in X_external
    gene_to_index = {gene: i for i, gene in enumerate(external_genes)}

    # Create a new matrix with shape (cells, model_genes), filled with zeros
    aligned_matrix = np.zeros((X_external.shape[0], len(model_genes)), dtype=np.float32)

    # For each gene expected by the model, fill in the matching values from X_external
    for j, gene in enumerate(model_genes):
        if gene in gene_to_index:
            aligned_matrix[:, j] = X_external[:, gene_to_index[gene]]
    return aligned_matrix


def load_and_align_data(count_path, gene_meta_path, model_genes, subset_size=20000):
    """
    Load and align a single dataset to match the model gene ordering.

    Parameters:
        count_path (str): Path to the .npz file containing gene expression counts.
        gene_meta_path (str): Path to the .pkl file containing gene metadata.
        model_genes (list[str]): Standardized list of model gene names.
        subset_size (int or None): Number of cells to load (use subset in Jupyter notebook or None for full dataset on server). 

    Returns:
        np.ndarray: Expression matrix aligned to model gene order (cells × genes).
    """
    X_raw = sp.load_npz(count_path)
 
    if subset_size is not None:
        X_raw = X_raw[:subset_size, :]

    gene_list = pd.read_pickle(gene_meta_path)["gene_symbol"].tolist()
    cleaned_genes, X_filtered = clean_and_filter_genes(gene_list, X_raw)
    X_filtered = X_filtered.toarray()
    X_aligned = align_input_matrix(X_filtered, cleaned_genes, model_genes)

    print(f"Loaded {count_path}")
    print("Raw shape:              ", X_raw.shape)
    print("Filtered shape:         ", X_filtered.shape)
    print("Aligned shape:          ", X_aligned.shape)
    print("Non-zero columns:       ", (X_aligned.sum(axis=0) > 0).sum())
    print("Model gene overlap:     ", len(set(cleaned_genes) & set(model_genes)))

    return X_aligned

# Optional
def load_multiple_datasets(count_paths, meta_paths, model_genes):
    """
    Load and align multiple datasets, then concatenate them along the cell axis.

    Parameters:
        count_paths (list[str]): Paths to .npz count files.
        meta_paths (list[str]): Paths to .pkl gene metadata files.
        model_genes (list[str]): Standardized list of model gene names.

    Returns:
        np.ndarray: Combined and aligned expression matrix (cells × genes).
    """
    aligned_matrices = []
    for count_path, gene_meta_path in zip(count_paths, meta_paths):
        aligned = load_and_align_data(count_path, gene_meta_path, model_genes)
        aligned_matrices.append(aligned)
    return np.vstack(aligned_matrices)

print("Defined helper functions")

Defined helper functions


In [32]:
# DEFINE FINE-TUNING AND EMBEDDINGS FUNCTIONS

def fine_tune_model(model, X_tensor, epochs=20, batch_size=512, lr=1e-4, l1_lambda=0.0):
    """
    Fine-tune a pre-trained autoencoder model on new input data.

    Parameters:
        model (Autoencoder): The model to be fine-tuned.
        X_tensor (torch.Tensor): Input data (cells × genes) as a float tensor.
        epochs (int): Number of training epochs.
        batch_size (int): Batch size for training.
        lr (float): Learning rate.
        l1_lambda (float): L1 penalty weight for sparsity regularization.

    Returns:
        None
    """
    loader = DataLoader(TensorDataset(X_tensor), batch_size=batch_size, shuffle=True)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    loss_fn  = nn.MSELoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for (x_batch,) in loader:
            optimizer.zero_grad()
            x_hat, _ = model(x_batch)
            loss = loss_fn(x_hat, x_batch) + l1_lambda * model.encoder.weight.abs().sum()
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                model.encoder.weight.clamp_(min=0.0)
            total_loss += loss.item()
        print(f"[Fine-Tune] Epoch {epoch+1} | Loss: {total_loss / len(loader):.4f}")


def get_embeddings(model, X_tensor):
    """
    Generate latent embeddings (encoder output) from input expression data.

    Parameters:
        model (Autoencoder): Trained autoencoder model.
        X_tensor (torch.Tensor): Input expression data (cells × genes).

    Returns:
        np.ndarray: Latent embeddings (cells × latent_dim) as NumPy array.
    """
    with torch.no_grad():
        _, H = model(X_tensor)
    return H.cpu().numpy()

print("Defined fine-tuning and embeddings functions")

Defined fine-tuning and embeddings functions


In [ ]:
# LOAD AND ALIGN EXTERNAL DATA

# Option 1: Load and align a SINGLE dataset
# Example usage (comment out if not needed):
count_path = "path_to_your_counts.npz"
meta_path  = "path_to_your_gene_metadata.pkl"

X = load_and_align_data(count_path, meta_path, model_genes, subset_size=20000)
X_tensor = torch.tensor(X, dtype=torch.float32)

print("Final X_tensor shape (single project):", X_tensor.shape)

# Option 2: Load and align MULTIPLE datasets
# Example usage (comment out if not needed):
count_paths = ["path_to_your_counts_1.npz", "path_to_your_counts_2.npz"]
meta_paths  = ["path_to_your_gene_metadata_1.pkl", "path_to_your_gene_metadata_2.pkl"]

X = load_multiple_datasets(count_paths, meta_paths, model_genes)
X_tensor = torch.tensor(X, dtype=torch.float32)

print("Final X_tensor shape (combined projects):", X_tensor.shape)

In [ ]:
# LOAD MODEL AND GET EMBEDDINGS

# Load the pre-trained model
model = Autoencoder(input_dim=len(model_genes), latent_dim=256)
model.load_state_dict(torch.load("/mnt/projects/debruinz_project/bisholea/capstone/60M Model/60M_model_state_dict_256.pth", map_location=torch.device("cpu")))

# Get embeddings BEFORE fine-tuning
embeddings_before_tuning = get_embeddings(model, X_tensor)

In [ ]:
# FINE-TUNE MODEL AND SAVE EMEBDDINGS

# Fine-tune the model on the new dataset
print("Fine-tuning model...")
fine_tune_model(model, X_tensor)

torch.save(model.state_dict(), "fine_tuned_model_state_dict.pth")
print("Saved fine-tuned model")

# Get embeddings AFTER fine-tuning
embeddings_after_tuning = get_embeddings(model, X_tensor)

# Save both sets of embeddings
np.save("embeddings_before_tuning.npy", embeddings_before_tuning)
np.save("embeddings_after_tuning.npy", embeddings_after_tuning)
print("Saved embeddings")

End of notebook